# 01-Nowicki: 数据读入 + 质量控制

> 作者已完成 QC + normalization。本 notebook 仅做数据读入、格式对齐与基础标记，不做任何过滤。

In [ ]:
# === PARAMS ===
MANIFEST_PATH = "data/nowicki/manifest.yaml"
OUTPUT_PATH = "results/01_nowicki_v1.h5ad"
QC_STRATEGY = "skip"
SCORE_CELL_CYCLE = True
RANDOM_SEED = 42
OUTPUT_VERSION = 1

In [ ]:
# === Setup：sys.path + 导入依赖 ===
import sys, os
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

from scrna_integration.io import read_with_manifest

In [ ]:
# 数据读入：通过 read_with_manifest 读入Nowicki数据集。
print(f"正在从 {MANIFEST_PATH} 读入...")
adata = read_with_manifest(MANIFEST_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"obs 列: {list(adata.obs.columns)}")
print(f"var 列: {list(adata.var.columns)}")


## 细胞周期评分

In [ ]:
# 细胞周期评分（Tirosh 2015 marker genes）
if SCORE_CELL_CYCLE:
    s_genes = ["MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7","DTL","PRIM1","UHRF1","MLF1IP","HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"]
    g2m_genes = ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","HN1","CDC20","TTK","CDC25C","KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"]
    sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)
    print("细胞周期评分完成")
    print(adata.obs["phase"].value_counts())
else:
    print("SCORE_CELL_CYCLE=False，跳过")


## 基因复杂度

In [ ]:
# 基因复杂度
adata.obs["log_complexity"] = np.log10(adata.obs["n_genes"] + 1) / np.log10(adata.obs["total_counts"] + 1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(adata.obs["log_complexity"].dropna(), bins=50, color="steelblue", edgecolor="white")
ax.set_xlabel("log10(n_genes+1) / log10(total_counts+1)")
ax.set_ylabel("细胞数")
ax.set_title("基因复杂度分布")
for pct in [1, 5, 25, 50, 75, 95, 99]:
    val = np.percentile(adata.obs["log_complexity"].dropna(), pct)
    ax.axvline(val, color="red", linestyle="--", alpha=0.3, linewidth=0.8)
plt.tight_layout()
fig.savefig("results/figures/01_nowicki_complexity.png", dpi=150, bbox_inches="tight")
plt.show()


## 原作者标注列确认

In [ ]:
# 确认原作者标注列已正确注入
annotation_cols = [c for c in adata.obs.columns if c.startswith("cell_type_original_")]
print("原作者标注列:")
for col in annotation_cols:
    vals = adata.obs[col].dropna().unique()
    print(f"  {col}: {len(vals)} 个唯一值: {sorted(vals)[:15]}...")


## QC 报告（跳过模式）

In [ ]:
# QC 报告
qc_report = {
    "strategy": "skip",
    "note": "作者已完成 basic_filter + doublet_removal + normalization；重新过滤会造成科学错误。",
    "cells_total": int(adata.n_obs),
    "cells_removed": 0,
    "pct_removed": 0.0,
    "cell_cycle_scored": SCORE_CELL_CYCLE,
}
adata.uns["qc_report_v1"] = qc_report
for k, v in qc_report.items():
    print(f"  {k}: {v}")


In [ ]:
# Checkpoint
import scipy.sparse as sp
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, f"adata.X invariant broken: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
adata.uns["stage"] = "01_qcd"
adata.uns["status"] = "experimental"
adata.uns["upstream"] = [MANIFEST_PATH]
adata.uns["version"] = f"v{OUTPUT_VERSION}"
os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"OK 写入 {OUTPUT_PATH}  ({adata.n_obs} cells x {adata.n_vars} genes)")
assert os.path.exists(OUTPUT_PATH), f"输出未找到: {OUTPUT_PATH}"
print(f"已校验: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")
del adata; gc.collect()
print("内存已释放。")
